# Dropout e batch normalization



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

In questo esercizio aggiungerai il dropout al modello *Spotify* dell’esercizio 4 e vedrai come la batch normalization può permettere di addestrare con successo modelli su dataset difficili.

Esegui la prossima cella per iniziare!


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex5 import *

Per prima cosa carica il dataset *Spotify*.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import GroupShuffleSplit

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import callbacks

spotify = pd.read_csv('../input/dl-course-data/spotify.csv')

X = spotify.copy().dropna()
y = X.pop('track_popularity')
artists = X['track_artist']

features_num = ['danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'duration_ms']
features_cat = ['playlist_genre']

preprocessor = make_column_transformer(
    (StandardScaler(), features_num),
    (OneHotEncoder(), features_cat),
)

def group_split(X, y, group, train_size=0.75):
    splitter = GroupShuffleSplit(train_size=train_size)
    train, test = next(splitter.split(X, y, groups=group))
    return (X.iloc[train], X.iloc[test], y.iloc[train], y.iloc[test])

X_train, X_valid, y_train, y_valid = group_split(X, y, artists)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
y_train = y_train / 100
y_valid = y_valid / 100

input_shape = [X_train.shape[1]]
print("Input shape: {}".format(input_shape))

# 1) Aggiungere il dropout al modello Spotify

Ecco l’ultimo modello dell’esercizio 4. Aggiungi due strati di dropout: uno dopo lo strato `Dense` con 128 unità e uno dopo quello con 64 unità. Imposta il tasso di dropout di entrambi a `0.3`.


In [ ]:
# SCRIVI QUI IL CODICE: aggiungi due layer di dropout al 30%, dopo i layer da 128 e 64 unità.
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# Controlla la risposta.
q_1.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_1.hint()
#q_1.solution()

Ora esegui la prossima cella per addestrare il modello e vedere l’effetto dell’aggiunta del dropout.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=50,
    verbose=0,
)
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()))

# 2) Valutare il dropout

Ricorda dall’esercizio 4 che questo modello tendeva all’overfitting intorno all’epoca 5. L’aggiunta del dropout sembra aver aiutato a prevenirlo questa volta?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_2.check()

Cambiamo argomento per esplorare come la batch normalization possa correggere problemi di addestramento.

Carica il dataset *Concrete*. Questa volta non eseguiremo alcuna standardizzazione: renderà molto più evidente l’effetto della batch normalization.


In [ ]:
import pandas as pd

concrete = pd.read_csv('../input/dl-course-data/concrete.csv')
df = concrete.copy()

df_train = df.sample(frac=0.7, random_state=0)
df_valid = df.drop(df_train.index)

X_train = df_train.drop('CompressiveStrength', axis=1)
X_valid = df_valid.drop('CompressiveStrength', axis=1)
y_train = df_train['CompressiveStrength']
y_valid = df_valid['CompressiveStrength']

input_shape = [X_train.shape[1]]

Esegui la cella seguente per addestrare la rete sui dati *Concrete* non standardizzati.


In [ ]:
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=input_shape),
    layers.Dense(512, activation='relu'),    
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])
model.compile(
    optimizer='sgd', # SGD è più sensibile alle differenze di scala.
    loss='mae',
    metrics=['mae'],
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=64,
    epochs=100,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.loc[0:, ['loss', 'val_loss']].plot()
print(("Minimum Validation Loss: {:0.4f}").format(history_df['val_loss'].min()))

Hai ottenuto un grafico vuoto? Il tentativo di addestrare questa rete su questo dataset di solito fallisce. Anche quando converge, grazie a un’inizializzazione fortunata dei pesi, tende a convergere a un numero molto grande.

# 3) Aggiungere strati di batch normalization

La batch normalization può aiutare a correggere problemi di questo tipo.

Aggiungi quattro strati `BatchNormalization`, uno prima di ciascuno strato dense. Ricorda di spostare l’argomento `input_shape` nel nuovo primo strato.


In [ ]:
# SCRIVI QUI IL CODICE: aggiungi un layer BatchNormalization prima di ogni layer Dense.
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=input_shape),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])

# Controlla la risposta.
q_3.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_3.hint()
#q_3.solution()

Esegui la prossima cella per vedere se la batch normalization ci permette di addestrare il modello.


In [ ]:
model.compile(
    optimizer='sgd',
    loss='mae',
    metrics=['mae'],
)
EPOCHS = 100
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=64,
    epochs=EPOCHS,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.loc[0:, ['loss', 'val_loss']].plot()
print(("Minimum Validation Loss: {:0.4f}").format(history_df['val_loss'].min()))

# 4) Valutare la batch normalization

L’aggiunta della batch normalization è stata utile?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_4.check()

# Prosegui

[**Crea reti neurali**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/tut6.html) per la classificazione binaria.
